### Dynamic notebook

In [0]:
dbutils.widgets.text("file_name","orders")

In [0]:
p_file_name = dbutils.widgets.get("file_name")

### Data Reading

In [0]:
# Auto Loader will automatically detect new files in that location and load them as a stream, using the schema stored at the checkpoint path.

# spark.readStream.format("cloudFiles")	        | "cloudFiles" enables "Databricks Auto Loader" to read streaming data from files in cloud storage.
#   .option("cloudFiles.format", "parquet")	    | Specifies the file format expected (Parquet in this case).
#   .option("cloudFiles.schemaLocation", ...)	| Tells Auto Loader where to store the schema so it can detect changes in future files (schema evolution).
#   .load(...)	                                | Loads files from the specified input path (in this case: a path in Azure Data Lake Storage Gen2).

df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemalocation", f"abfss://bronze@databricksuk2025.dfs.core.windows.net/checkpoint_{p_file_name}")\
    .load(f"abfss://source@databricksuk2025.dfs.core.windows.net/{p_file_name}")

In [0]:
#df.writeStream.format("parquet")	    | Writes the streaming data in Parquet format.
#   .outputMode("append")	            | Appends only new records (typical for file-based streaming).
#   .option("checkpointLocation", ...)	| Stores the streaming checkpoint data, so the process can resume if interrupted (exactly-once semantics).
#   .option("path", ...)	            | The target path where the processed Parquet files will be saved (i.e., your Bronze layer).
#   .trigger(once=True)	                | Runs the streaming job just once, processing all available files like a batch job (very useful in Auto Loader for controlled ingestion).
#   .start()                        	| Starts the streaming write process.

df.writeStream.format("parquet")\
    .outputMode("append")\
    .option("checkpointLocation",f"abfss://bronze@databricksuk2025.dfs.core.windows.net/checkpoint_{p_file_name}")\
    .option("path",f"abfss://bronze@databricksuk2025.dfs.core.windows.net/{p_file_name}")\
    .trigger(once=True)\
    .start()

In [0]:
df = spark.read.format("parquet")\
    .load(f"abfss://bronze@databricksuk2025.dfs.core.windows.net/{p_file_name}")
display(df)